# BB Mean Reversion Strategy — Интерактивный ноутбук

**Стратегия:** Bollinger Bands + RSI + ADX на Binance Futures (1h)  
**Логика входа:** `Close < BB_lower AND RSI < 30 AND ADX < 30` (LONG) / зеркально (SHORT)  
**Логика выхода:** цена достигла BB_middle (BB_MID) или сработал стоп (ATR × mult)

---
**Порядок запуска ячеек:**
1. ⚙️ **Настройки** — задайте символ/топ-N и параметры
2. 🔍 **Список монет** — CoinGecko + Binance Futures фильтр
3. 📥 **Загрузка свечей** — скачать OHLCV с Binance
4. 📊 **Бэктест** — запуск стратегии на исторических данных
5. 🔬 **Оптимизация** — Optuna поиск лучших параметров

In [ ]:
# Одноразовая установка (раскомментировать если пакет не установлен)
import subprocess, sys

def _install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], check=True)

_install('nest_asyncio')   # разрешает asyncio.run() внутри Jupyter
_install('ipywidgets')     # прогресс-бар Optuna в Jupyter (устраняет TqdmWarning)
_install('scikit-learn')   # графики важности параметров Optuna (plot_param_importances)

---
## ⚙️ Ячейка 1 — Настройки
Единственная точка конфигурации. Все последующие ячейки читают переменные `_NB_*`.

In [ ]:
import sys
sys.path.insert(0, '..')  # project root → можно импортировать feed, strategy, optimization

# ── РЕЖИМ: одна монета ИЛИ топ-N ────────────────────────────────────────────
_NB_SYMBOL = None          # символ монеты ('BTCUSDT') или None для режима топ-N
_NB_TOP_N  = 5             # сколько монет брать из топа по капитализации (только при _NB_SYMBOL=None)

# ── ПЕРИОД И ТАЙМФРЕЙМ ──────────────────────────────────────────────────────
_NB_INTERVAL   = '15m'         # таймфрейм свечей: 1m, 5m, 15m, 30m, 1h, 4h, 1d
_NB_START_DATE = '2026-01-01'  # начальная дата в формате yyyy-mm-dd
_NB_END_DATE   = '2026-05-19'  # конечная дата в формате yyyy-mm-dd
_NB_KLINES_DIR = '../klines'  # папка для сохранения/чтения CSV-файлов со свечами

# ── ПАРАМЕТРЫ СТРАТЕГИИ ──────────────────────────────────────────────────────
_NB_STRATEGY_PARAMS = {
    'bb_len':        20,    # период Bollinger Bands (количество свечей)
    'bb_std':        2.0,   # множитель стандартного отклонения для ширины полос BB
    'rsi_len':       14,    # период RSI (количество свечей)
    'rsi_lower':     25,    # уровень перепроданности RSI: вход в LONG, если RSI < этого значения
    'rsi_upper':     75,    # уровень перекупленности RSI: вход в SHORT, если RSI > этого значения
    'adx_len':       14,    # период ADX (количество свечей)
    'adx_threshold': 30,    # максимальный ADX для входа: торгуем только на боковике (ADX < порога)
    'sl_atr_mult':   2.0,   # стоп-лосс = entry ± ATR × этот множитель
    'trade_usdt':    150,   # размер позиции в USDT на одну сделку
}
_NB_INITIAL_CASH = 1000.0  # стартовый капитал для бэктеста (USD)

# ── ПАРАМЕТРЫ ОПТИМИЗАЦИИ ────────────────────────────────────────────────────
_NB_N_TRIALS      = 100    # число попыток Optuna: 50–200 для быстрого теста, 500–1000 для точного
_NB_N_JOBS        = 1      # параллельных процессов (1 = последовательно; >1 требует multiprocessing)
_NB_OBJECTIVE     = 'composite'  # целевая функция: composite / sharpe / sortino / calmar / risk_adjusted
_NB_MIN_TRADES    = 10     # минимум сделок в trial'е (меньше — trial отклоняется как невалидный)
_NB_OUT_OF_SAMPLE = 0.20   # доля данных для Out-of-Sample теста: 0.20 = последние 20%
_NB_WALK_FORWARD  = True   # включить walk-forward валидацию (True) или ограничиться IS/OOS (False)
_NB_WF_FOLDS      = 5      # число фолдов для walk-forward (рекомендуется 3–10)
_NB_SAMPLER       = 'tpe'  # алгоритм поиска: tpe (байесовский), cmaes (эволюционный), nsga2 (многоцелевой), random
_NB_OPT_SYMBOL    = None   # символ для оптимизации: None → берётся _NB_SYMBOL или первый из списка

# ── СЛУЖЕБНЫЕ (заполняются ячейками 2 и 3) ───────────────────────────────────
_NB_SYMBOLS_LIST = []
_NB_COINS_DF     = None
_NB_DATA         = {}

# ── ИТОГ ─────────────────────────────────────────────────────────────────────
if _NB_SYMBOL:
    _mode = 'Одна монета: ' + _NB_SYMBOL
else:
    _mode = 'Топ-' + str(_NB_TOP_N)

print('✓ Настройки загружены')
print(f'  Режим     : {_mode}')
print(f'  Интервал  : {_NB_INTERVAL}  |  {_NB_START_DATE} -> {_NB_END_DATE}')
print(f'  Параметры : {_NB_STRATEGY_PARAMS}')
print(f'  Капитал   : ${_NB_INITIAL_CASH:.0f}  |  Opt trials: {_NB_N_TRIALS}  obj={_NB_OBJECTIVE}')

---
## 🔍 Ячейка 2 — Формирование списка монет
Получает топ-N монет с CoinGecko и фильтрует по доступным парам Binance Futures (USDT).
При `_NB_SYMBOL != None` просто подтверждает одну монету.

In [3]:
import feed.candles as _candles_mod
import config.settings as _cfg
import pandas as pd
from IPython.display import display

if _NB_SYMBOL is not None:
    # ── Режим одной монеты ──────────────────────────────────────────────────
    _NB_SYMBOLS_LIST = [_NB_SYMBOL]
    print(f'✓ Режим одной монеты: {_NB_SYMBOL}')

else:
    # ── Режим топ-N: CoinGecko → Binance Futures фильтр ────────────────────
    # Monkey-patch: feed.candles захватил top_by_cap при импорте,
    # нужно обновить переменную прямо в модуле
    _cfg.top_by_cap = _NB_TOP_N
    _candles_mod.top_by_cap = _NB_TOP_N

    print(f'Получение топ-{_NB_TOP_N} монет (CoinGecko + Binance Futures)...')
    coins = _candles_mod.get_binance_top_by_cap()

    if not coins:
        raise RuntimeError('Не удалось получить список монет. Проверьте интернет.')

    _NB_SYMBOLS_LIST = [c['symbol'].upper() + 'USDT' for c in coins]

    _NB_COINS_DF = pd.DataFrame([{
        'Rank':            c.get('market_cap_rank', i),
        'Symbol':          c['symbol'].upper() + 'USDT',
        'Name':            c['name'],
        'Market Cap ($B)': round(c['market_cap'] / 1e9, 3),
        'Price ($)':       c['current_price'],
        '24h Change %':    round(c.get('price_change_percentage_24h') or 0.0, 2),
    } for i, c in enumerate(coins, 1)])

    try:
        display(
            _NB_COINS_DF.style
            .format({
                'Market Cap ($B)': '{:.3f}',
                'Price ($)':       '{:.4f}',
                '24h Change %':    '{:+.2f}',
            })
            .set_caption(f'Топ-{_NB_TOP_N} по рыночной капитализации (Binance Futures USDT)')
        )
    except Exception:
        display(_NB_COINS_DF)

_preview = ', '.join(_NB_SYMBOLS_LIST[:10])
_suffix  = '...' if len(_NB_SYMBOLS_LIST) > 10 else ''
print(f'\n✓ Список монет готов ({len(_NB_SYMBOLS_LIST)} шт.): {_preview}{_suffix}')

Получение топ-5 монет (CoinGecko + Binance Futures)...
Запрашиваем данные с CoinGecko...
✓ Получено 100 монет с CoinGecko
Запрашиваем данные с Binance...
✓ 648 активных USDT-пар на Binance Futures
✓ Найдено 55 общих монет


,Rank,Symbol,Name,Market Cap ($B),Price ($),24h Change %
0,1,BTCUSDT,Bitcoin,1253.459,62508.0000,+1.70
1,2,ETHUSDT,Ethereum,212.305,1758.3200,+3.52
2,4,BNBUSDT,BNB,77.053,571.6900,+2.56
3,5,USDCUSDT,USDC,72.981,0.9999,+0.01
4,6,XRPUSDT,XRP,70.233,1.1300,+4.42



✓ Список монет готов (5 шт.): BTCUSDT, ETHUSDT, BNBUSDT, USDCUSDT, XRPUSDT


---
## 📥 Ячейка 3 — Загрузка свечей
Параллельная async-загрузка OHLCV с Binance Futures API → сохраняет CSV в `klines/`.

> **Если данные уже скачаны** — раскомментируйте блок «Вариант 2» внизу ячейки и закомментируйте «Вариант 1».

In [4]:
import asyncio
import time
import nest_asyncio
import pandas as pd
from IPython.display import display
import feed.candles as _candles_mod

nest_asyncio.apply()  # разрешает asyncio.run() внутри уже запущенного Jupyter event loop

# ── Вариант 1: скачать с Binance ─────────────────────────────────────────────
print(f'Загрузка свечей: {len(_NB_SYMBOLS_LIST)} символов  [{_NB_INTERVAL}]  '
      f'{_NB_START_DATE} -> {_NB_END_DATE}')
print(f'Директория: {_NB_KLINES_DIR}\n')

_t0 = time.monotonic()

_NB_DATA = asyncio.run(
    _candles_mod.download_many_symbols_async(
        _NB_SYMBOLS_LIST,
        _NB_INTERVAL,
        start_date=_NB_START_DATE,
        end_date=_NB_END_DATE,
        filepath=_NB_KLINES_DIR,
        save=True,
    )
)

_elapsed = time.monotonic() - _t0

# Сводная таблица результатов
_summary_rows = []
for _sym in _NB_SYMBOLS_LIST:
    _df = _NB_DATA.get(_sym)
    if _df is not None and not _df.empty:
        _summary_rows.append({
            'Symbol': _sym,
            'Свечей': len(_df),
            'С':      str(_df['Timestamp'].min())[:10],
            'По':     str(_df['Timestamp'].max())[:10],
            'Статус': 'OK',
        })
    else:
        _summary_rows.append({'Symbol': _sym, 'Свечей': 0, 'С': '-', 'По': '-', 'Статус': 'FAILED'})

_summary_df = pd.DataFrame(_summary_rows)
display(_summary_df)

_ok  = (_summary_df['Статус'] == 'OK').sum()
_err = (_summary_df['Статус'] == 'FAILED').sum()
print(f'\n✓ Загружено за {_elapsed:.1f}с  |  OK: {_ok}  |  Ошибок: {_err}')


# ── Вариант 2: читать из существующих CSV (без повторной загрузки) ────────────
# Раскомментируйте этот блок и закомментируйте Вариант 1 выше
#
# _NB_DATA = {}
# for _sym in _NB_SYMBOLS_LIST:
#     try:
#         _df = _candles_mod.read_df_from_csv(
#             _sym, _NB_INTERVAL, _NB_START_DATE, _NB_END_DATE, _NB_KLINES_DIR
#         )
#         _NB_DATA[_sym] = _df if (_df is not None and not _df.empty) else None
#     except Exception:
#         _NB_DATA[_sym] = None
#
# _loaded = sum(1 for v in _NB_DATA.values() if v is not None)
# print(f'✓ Прочитано из CSV: {_loaded} / {len(_NB_SYMBOLS_LIST)}')


Загрузка свечей: 5 символов  [15m]  2026-01-01 -> 2026-05-19
Директория: klines

✓ USDCUSDT: 13,345 свечей
  Сохранён: klines\USDCUSDT_15m_2026-01-01_2026-05-19.csv  (13,345 строк, 1042.0 KB)
✓ BNBUSDT: 13,345 свечей
  Сохранён: klines\BNBUSDT_15m_2026-01-01_2026-05-19.csv  (13,345 строк, 1135.5 KB)
✓ BTCUSDT: 13,345 свечей
  Сохранён: klines\BTCUSDT_15m_2026-01-01_2026-05-19.csv  (13,345 строк, 1232.1 KB)
✓ ETHUSDT: 13,345 свечей
  Сохранён: klines\ETHUSDT_15m_2026-01-01_2026-05-19.csv  (13,345 строк, 1199.0 KB)
✓ XRPUSDT: 13,345 свечей
  Сохранён: klines\XRPUSDT_15m_2026-01-01_2026-05-19.csv  (13,345 строк, 1070.4 KB)


,Symbol,Свечей,С,По,Статус
0,BTCUSDT,13345,2025-12-31,2026-05-19,OK
1,ETHUSDT,13345,2025-12-31,2026-05-19,OK
2,BNBUSDT,13345,2025-12-31,2026-05-19,OK
3,USDCUSDT,13345,2025-12-31,2026-05-19,OK
4,XRPUSDT,13345,2025-12-31,2026-05-19,OK



✓ Загружено за 22.6с  |  OK: 5  |  Ошибок: 0


---
## 📊 Ячейка 4 — Бэктест
Запускает стратегию на исторических данных с параметрами из ячейки 1.
- **Одна монета:** полная таблица метрик + интерактивный Plotly-график
- **Несколько монет:** сводная таблица с цветовым градиентом (Sharpe, Return, MaxDD)

In [5]:
import warnings
import pandas as pd
import backtrader as bt
from IPython.display import display
from optimization.optimizer import run_backtest_with_params, suppress_output
from strategy.backtest import SimpleMeanReversionStrategy, plot_tradingview
import feed.candles as _candles_mod

warnings.filterwarnings('ignore')


def _load_df_bt(symbol):
    """Вернуть DataFrame из _NB_DATA или прочитать из CSV (fallback)."""
    df = _NB_DATA.get(symbol) if _NB_DATA else None
    if df is None or df.empty:
        try:
            df = _candles_mod.read_df_from_csv(
                symbol, _NB_INTERVAL, _NB_START_DATE, _NB_END_DATE, _NB_KLINES_DIR
            )
        except Exception:
            df = None
    return df


def _run_for_chart(df, params, initial_cash):
    """Бэктест с захватом trades_log и equity для plot_tradingview."""
    df_bt = df.copy()
    df_bt.rename(columns={
        'Timestamp': 'datetime', 'Open': 'open',
        'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume',
    }, inplace=True)
    df_bt['datetime'] = pd.to_datetime(df_bt['datetime'])
    df_bt.set_index('datetime', inplace=True)
    df_bt = df_bt.iloc[:-1]  # последняя незакрытая свеча

    cerebro = bt.Cerebro()
    cerebro.adddata(bt.feeds.PandasData(dataname=df_bt))
    cerebro.addstrategy(SimpleMeanReversionStrategy, **params)
    cerebro.broker.setcash(initial_cash)
    cerebro.broker.setcommission(commission=0.001)

    with suppress_output():
        results = cerebro.run(maxcpus=1)

    strat = results[0]
    trades_log = strat.trades_log

    equity = None
    if trades_log:
        _pnl = pd.DataFrame(trades_log)
        _pnl['Exit Time'] = pd.to_datetime(_pnl['Exit Time'])
        _pnl = _pnl.sort_values('Exit Time')
        equity = pd.Series(
            (initial_cash + _pnl['PnL (USD)'].cumsum()).values,
            index=_pnl['Exit Time'].values,
        )
    return df_bt, trades_log, equity


# ── Запуск бэктестов ──────────────────────────────────────────────────────────
_bt_rows = []
for _sym in _NB_SYMBOLS_LIST:
    _df = _load_df_bt(_sym)
    if _df is None or _df.empty:
        print(f'[пропуск] {_sym}: нет данных')
        continue

    with suppress_output():
        _m = run_backtest_with_params(_df, _NB_STRATEGY_PARAMS, _NB_INITIAL_CASH)

    _bt_rows.append({
        'Symbol':         _sym,
        'Сделок':         _m['total_trades'],
        'Win Rate %':     round(_m['win_rate'] * 100, 1),
        'Return %':       round(_m['return_pct'], 2),
        'Max DD %':       round(_m['max_dd'] * 100, 2),
        'Sharpe':         round(_m['sharpe'], 3),
        'Sortino':        round(_m['sortino'], 3),
        'Profit Factor':  round(_m['profit_factor'], 3),
        'SQN':            round(_m['sqn'], 3),
        'Final ($)':      round(_m['final_value'], 2),
    })

_NB_BACKTEST_RESULTS = pd.DataFrame(_bt_rows)

# ── Отображение результатов ───────────────────────────────────────────────────
if len(_NB_SYMBOLS_LIST) == 1:
    # Одна монета: транспонированная таблица + Plotly-график
    _sym = _NB_SYMBOLS_LIST[0]
    print(f'── Результаты бэктеста: {_sym} ──')
    if not _NB_BACKTEST_RESULTS.empty:
        display(_NB_BACKTEST_RESULTS.T.rename(columns={0: 'Значение'}))

        _df_raw = _load_df_bt(_sym)
        _df_chart, _tlog, _equity = _run_for_chart(
            _df_raw, _NB_STRATEGY_PARAMS, _NB_INITIAL_CASH
        )
        print(f'\nГрафик: {len(_tlog)} сделок')
        plot_tradingview(
            _df_chart, _tlog, _equity,
            symbol=_sym,
            tf=_NB_INTERVAL,
            bb_len=_NB_STRATEGY_PARAMS['bb_len'],
            bb_std_val=_NB_STRATEGY_PARAMS['bb_std'],
            rsi_len=_NB_STRATEGY_PARAMS['rsi_len'],
            adx_len=_NB_STRATEGY_PARAMS['adx_len'],
        )
    else:
        print('Нет результатов. Проверьте данные.')

else:
    # Несколько монет: сводная таблица, сортировка по Sharpe
    _NB_BACKTEST_RESULTS = _NB_BACKTEST_RESULTS.sort_values('Sharpe', ascending=False)
    _fmt = {
        'Win Rate %':    '{:.1f}',
        'Return %':      '{:.2f}',
        'Max DD %':      '{:.2f}',
        'Sharpe':        '{:.3f}',
        'Sortino':       '{:.3f}',
        'Profit Factor': '{:.3f}',
        'SQN':           '{:.3f}',
        'Final ($)':     '{:.2f}',
    }
    try:
        display(
            _NB_BACKTEST_RESULTS.style
            .background_gradient(subset=['Sharpe', 'Return %', 'Win Rate %'], cmap='RdYlGn')
            .background_gradient(subset=['Max DD %'], cmap='RdYlGn_r')
            .format(_fmt)
            .set_caption('Результаты бэктеста (отсортировано по Sharpe ↓)')
        )
    except Exception:
        display(_NB_BACKTEST_RESULTS.style.format(_fmt))

print(f'\n✓ Бэктест завершён: {len(_bt_rows)} символов')

C:\Python_Projects\WORK\ALGOROBOTIX\1 (BB_mean_reversion_rsi_adx)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Symbol,Сделок,Win Rate %,Return %,Max DD %,Sharpe,Sortino,Profit Factor,SQN,Final ($)
2,BNBUSDT,51,45.1,-1.73,3.20,-1.687,-10.363,0.537,-1.997,982.68
1,ETHUSDT,49,49.0,-2.18,3.48,-2.231,-6.590,0.531,-1.830,978.21
4,XRPUSDT,45,44.4,-2.38,2.54,-2.253,-8.854,0.490,-2.069,976.25
0,BTCUSDT,52,50.0,-2.30,2.39,-3.057,-10.923,0.415,-2.630,976.98
3,USDCUSDT,34,0.0,-1.01,1.01,-7.462,-315.338,0.000,-117.668,989.87



✓ Бэктест завершён: 5 символов


---
## 🔬 Ячейка 5 — Оптимизация параметров
Optuna (TPE/CMA-ES/NSGA-II) ищет оптимальные параметры стратегии.
- Разбивает данные на **In-Sample (train)** и **Out-of-Sample (OOS)**
- Опционально: **Walk-Forward** валидация
- Вердикт по **переобучению**: OK / BORDERLINE / OVERFIT

> Символ для оптимизации: `_NB_OPT_SYMBOL` (или `_NB_SYMBOL`, или первый из списка)

In [6]:
import argparse
import warnings
import pandas as pd
import optuna
from datetime import datetime
from optuna.pruners import MedianPruner
from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from optimization.optimizer import (
    run_backtest_with_params,
    make_objective,
    walk_forward_validation,
    detect_overfitting,
    build_sampler,
    suppress_output,
    print_comparison_table,
)
import feed.candles as _candles_mod
from config.settings import bb_lenth as _def_bb_len, bb_std as _def_bb_std
from config.settings import rsi_len as _def_rsi_len, adx_len as _def_adx_len

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


def _load_df_opt(symbol):
    """Загрузить данные из _NB_DATA или из CSV (fallback)."""
    df = _NB_DATA.get(symbol) if _NB_DATA else None
    if df is None or df.empty:
        try:
            df = _candles_mod.read_df_from_csv(
                symbol, _NB_INTERVAL, _NB_START_DATE, _NB_END_DATE, _NB_KLINES_DIR
            )
        except Exception:
            df = None
    return df


# ── Выбор символа для оптимизации ────────────────────────────────────────────
_opt_symbol = _NB_OPT_SYMBOL or _NB_SYMBOL or (_NB_SYMBOLS_LIST[0] if _NB_SYMBOLS_LIST else None)
if _opt_symbol is None:
    raise ValueError('Символ не определён. Запустите ячейку 2 или задайте _NB_OPT_SYMBOL.')

print('=' * 62)
print(f'  ОПТИМИЗАЦИЯ: {_opt_symbol}')
print(f'  Trials: {_NB_N_TRIALS}  |  Objective: {_NB_OBJECTIVE}  |  OOS: {_NB_OUT_OF_SAMPLE:.0%}')
print(f'  Walk-forward: {_NB_WALK_FORWARD}  ({_NB_WF_FOLDS} фолдов)')
print('=' * 62)

# ── Загрузка и разбивка данных ────────────────────────────────────────────────
_df_full = _load_df_opt(_opt_symbol)
if _df_full is None or len(_df_full) < 200:
    raise ValueError(f'Недостаточно данных для {_opt_symbol}. Запустите ячейку 3.')

_df_full = _df_full[
    (_df_full['Timestamp'] >= pd.to_datetime(_NB_START_DATE)) &
    (_df_full['Timestamp'] <= pd.to_datetime(_NB_END_DATE))
].reset_index(drop=True)

_split_idx = int(len(_df_full) * (1.0 - _NB_OUT_OF_SAMPLE))
_df_train  = _df_full.iloc[:_split_idx].reset_index(drop=True)
_df_oos    = _df_full.iloc[_split_idx:].reset_index(drop=True)

_t_from   = str(_df_full['Timestamp'].iloc[0])[:10]
_t_to     = str(_df_full['Timestamp'].iloc[-1])[:10]
_oos_from = str(_df_oos['Timestamp'].iloc[0])[:10]
print(f'\nДанных всего: {len(_df_full)} баров  ({_t_from} .. {_t_to})')
print(f'Train: {len(_df_train)} баров  |  OOS: {len(_df_oos)} баров  (с {_oos_from})')

# ── Baseline: дефолтные параметры (для сравнения) ────────────────────────────
_default_params = {
    'bb_len': _def_bb_len, 'bb_std': _def_bb_std,
    'rsi_len': _def_rsi_len, 'rsi_lower': 30, 'rsi_upper': 70,
    'adx_len': _def_adx_len, 'adx_threshold': 30, 'sl_atr_mult': 2.5,
}
with suppress_output():
    _default_m = run_backtest_with_params(_df_train, _default_params, _NB_INITIAL_CASH)

print(f'\nДефолт IS: Return={_default_m["return_pct"]:.1f}%  '
      f'Sharpe={_default_m["sharpe"]:.3f}  '
      f'Trades={_default_m["total_trades"]}  '
      f'MaxDD={_default_m["max_dd"]:.2%}')

# ── Настройка аргументов для оптимизатора ────────────────────────────────────
_opt_args = argparse.Namespace(
    symbol=_opt_symbol,
    trials=_NB_N_TRIALS,
    jobs=_NB_N_JOBS,
    sampler=_NB_SAMPLER,
    objective=_NB_OBJECTIVE,
    walk_forward=_NB_WALK_FORWARD,
    wf_folds=_NB_WF_FOLDS,
    out_of_sample=_NB_OUT_OF_SAMPLE,
    initial_cash=_NB_INITIAL_CASH,
    min_trades=_NB_MIN_TRADES,
    no_plots=False,
    study_name=None,
    storage=None,
)

# ── Создание и запуск Optuna study ───────────────────────────────────────────
_study_name = f"{_opt_symbol}_{_NB_OBJECTIVE}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
_NB_OPT_STUDY = optuna.create_study(
    study_name=_study_name,
    direction='maximize',
    sampler=build_sampler(_NB_SAMPLER),
    pruner=MedianPruner(n_startup_trials=15, n_warmup_steps=5),
)

print(f'\nЗапуск оптимизации ({_NB_N_TRIALS} trials)...')
_NB_OPT_STUDY.optimize(
    make_objective(_df_train, _opt_args),
    n_trials=_NB_N_TRIALS,
    n_jobs=_NB_N_JOBS,
    show_progress_bar=True,
    gc_after_trial=True,
)

if not _NB_OPT_STUDY.best_trial:
    raise RuntimeError('Все trials были pruned. Увеличьте n_trials или уменьшите _NB_MIN_TRADES.')

_NB_BEST_PARAMS = _NB_OPT_STUDY.best_trial.params
print(f'\n✓ Лучший trial #{_NB_OPT_STUDY.best_trial.number}  Score: {_NB_OPT_STUDY.best_trial.value:.4f}')

# ── Финальные метрики на train и OOS ─────────────────────────────────────────
with suppress_output():
    _best_train_m = run_backtest_with_params(_df_train, _NB_BEST_PARAMS, _NB_INITIAL_CASH)
    _NB_OOS_M     = run_backtest_with_params(_df_oos,   _NB_BEST_PARAMS, _NB_INITIAL_CASH)

# ── Walk-forward валидация ────────────────────────────────────────────────────
_wf_df = pd.DataFrame()
if _NB_WALK_FORWARD and len(_df_full) >= 400:
    print(f'\nWalk-forward ({_NB_WF_FOLDS} фолдов)...')
    _wf_df = walk_forward_validation(
        _df_full, _NB_BEST_PARAMS,
        n_folds=_NB_WF_FOLDS,
        test_ratio=_NB_OUT_OF_SAMPLE,
        initial_cash=_NB_INITIAL_CASH,
    )
    if not _wf_df.empty:
        _wf_cols = ['fold', 'return_pct', 'sharpe', 'win_rate', 'total_trades']
        _extra   = [c for c in ['test_start', 'test_end'] if c in _wf_df.columns]
        display(
            _wf_df[_extra + _wf_cols]
            .rename(columns={'return_pct': 'Return%', 'win_rate': 'WinRate', 'total_trades': 'Trades'})
        )

# ── Проверка переобучения ─────────────────────────────────────────────────────
_overfit = detect_overfitting(_best_train_m, _NB_OOS_M, _wf_df)
_vc = {'OK': '#00c853', 'BORDERLINE': '#ff9100', 'OVERFIT': '#f44336'}
_v  = _overfit['verdict']
display(HTML(f'<h3>Переобучение: <span style="color:{_vc[_v]}">{_v}</span></h3>'))

for _flag in _overfit['flags']:
    print(f'  [!] {_flag}')
if not _overfit['flags']:
    print('  [OK] Признаков переобучения не обнаружено')

# ── Сравнительная таблица: Default vs Best IS vs OOS ─────────────────────────
print(f'\n{"─" * 62}')
print_comparison_table(_default_params, _NB_BEST_PARAMS, _default_m, _best_train_m, _NB_OOS_M)

# ── Таблица лучших параметров ─────────────────────────────────────────────────
print(f'\n{"─" * 62}')
print('Оптимизированные параметры:')
_params_df = pd.DataFrame({
    'Параметр':  list(_NB_BEST_PARAMS.keys()),
    'Default':   [_default_params.get(k, '-') for k in _NB_BEST_PARAMS],
    'Optimized': list(_NB_BEST_PARAMS.values()),
}).set_index('Параметр')
display(_params_df)

# ── Optuna графики ────────────────────────────────────────────────────────────
print('\nГрафики Optuna:')
try:
    optuna.visualization.plot_optimization_history(_NB_OPT_STUDY).show()
    optuna.visualization.plot_param_importances(_NB_OPT_STUDY).show()
    optuna.visualization.plot_parallel_coordinate(_NB_OPT_STUDY).show()
except Exception as _e:
    print(f'  [warn] Не удалось построить графики Optuna: {_e}')

# ── Walk-forward бар-чарт ─────────────────────────────────────────────────────
if not _wf_df.empty and 'return_pct' in _wf_df.columns:
    _fig_wf = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Return % по фолдам', 'Sharpe по фолдам'),
    )
    _green, _red = '#26a69a', '#ef5350'
    _fig_wf.add_trace(go.Bar(
        x=_wf_df['fold'].astype(str),
        y=_wf_df['return_pct'],
        marker_color=[_green if v >= 0 else _red for v in _wf_df['return_pct']],
        name='Return %',
    ), row=1, col=1)
    if 'sharpe' in _wf_df.columns:
        _fig_wf.add_trace(go.Bar(
            x=_wf_df['fold'].astype(str),
            y=_wf_df['sharpe'],
            marker_color=[_green if v >= 0 else _red for v in _wf_df['sharpe']],
            name='Sharpe',
        ), row=2, col=1)
    _fig_wf.update_layout(
        title=f'Walk-Forward Validation -- {_opt_symbol}',
        height=500,
        template='plotly_dark',
        showlegend=False,
    )
    _fig_wf.show()

print(f'\n✓ Оптимизация завершена.')
print(f'  Лучшие параметры -> _NB_BEST_PARAMS')
print(f'  Study             -> _NB_OPT_STUDY')
print(f'  OOS метрики       -> _NB_OOS_M')


  ОПТИМИЗАЦИЯ: BTCUSDT
  Trials: 100  |  Objective: composite  |  OOS: 20%
  Walk-forward: True  (5 фолдов)

Данных всего: 13249 баров  (2026-01-01 .. 2026-05-19)
Train: 10599 баров  |  OOS: 2650 баров  (с 2026-04-21)

Дефолт IS: Return=1.1%  Sharpe=0.587  Trades=57  MaxDD=2.54%

Запуск оптимизации (100 trials)...


Best trial: 91. Best value: 1.88075: 100%|██████████| 100/100 [04:01<00:00,  2.41s/it] 



✓ Лучший trial #91  Score: 1.8808

Walk-forward (5 фолдов)...


,test_start,test_end,fold,Return%,sharpe,WinRate,Trades
0,2026-02-25 04:45:00,2026-03-24 19:00:00,0,-0.1873,-1.8626,0.4000,5
1,2026-03-11 00:00:00,2026-04-07 14:15:00,1,-0.0536,-0.2304,0.3333,6
2,2026-03-24 19:15:00,2026-04-21 09:30:00,2,2.0937,3.9837,0.5000,4
3,2026-04-07 14:30:00,2026-05-05 04:45:00,3,-0.5550,-4.1187,0.3333,3
4,2026-04-21 09:45:00,2026-05-19 00:00:00,4,-0.0106,-0.5854,0.3333,3


  [!] Sharpe деградировал на 129% (IS→OOS), порог 50%
  [!] WinRate деградировал на 42% (IS→OOS), порог 30%
  [!] Return% деградировал на 100% (IS→OOS), порог 50%
  [!] Sharpe стал отрицательным на OOS — сильный сигнал переобучения
  [!] Высокая вариативность Sharpe по фолдам (std=2.96)
  [!] Только 1/5 walk-forward фолдов прибыльны

──────────────────────────────────────────────────────────────

СРАВНЕНИЕ: ДЕФОЛТНЫЕ vs ЛУЧШИЕ ПАРАМЕТРЫ
Параметр                   Дефолт      Оптимум
---------------------------------------------
  adx_len                      14           22 ◄
  adx_threshold                30           22 ◄
  bb_len                       20           43 ◄
  bb_std                      2.0         2.25 ◄
  rsi_len                      14           28 ◄
  rsi_lower                    30           36 ◄
  rsi_upper                    70           71 ◄
  sl_atr_mult                 2.5         4.25 ◄

Метрика                   Дефолт (IS)     Оптим (IS)    Оптим (OOS)
-----

,Default,Optimized
Параметр,,
bb_len,20.0,43.00
bb_std,2.0,2.25
rsi_len,14.0,28.00
rsi_lower,30.0,36.00
rsi_upper,70.0,71.00
adx_len,14.0,22.00
adx_threshold,30.0,22.00
sl_atr_mult,2.5,4.25



Графики Optuna:


  [warn] Не удалось построить графики Optuna: Tried to import 'sklearn' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'sklearn'.



✓ Оптимизация завершена.
  Лучшие параметры -> _NB_BEST_PARAMS
  Study             -> _NB_OPT_STUDY
  OOS метрики       -> _NB_OOS_M
